# 04 — Benchmarking & Profiling

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/winstonsmith1897/DantinoX/blob/main/docs/notebooks/04_benchmarking.ipynb)

Covers:
1. Analytical FLOPs estimation (`count_flops` / `dx.profile`)
2. Wall-clock latency measurement (`LatencyTracker`)
3. Full benchmark suite across multiple configs (`BenchmarkSuite.default()`)
4. Comparing attention variants (MHA vs. GQA vs. MLA)
5. Parameter count via `paradigm.num_parameters()`
6. Visualization (`Visualizer`)

**Runtime**: GPU (T4 or better)

In [ ]:
!pip install -q git+https://github.com/winstonsmith1897/DantinoX.git#egg=dantinox[all]

In [ ]:
import dantinox as dx
from dantinox.benchmarking import BenchmarkSuite, BenchmarkConfig
from dantinox.profiling import LatencyTracker, count_flops
from flax import nnx
import jax
import jax.numpy as jnp

cfg      = dx.ModelConfig(dim=256, n_heads=4, head_size=64, num_blocks=4,
                           vocab_size=2000, causal=True)
paradigm = dx.ARParadigm(cfg)
model    = paradigm.build_model(nnx.Rngs(0))
print(f'Parameters: {paradigm.num_parameters(model):,}')

## 1. Analytical FLOPs

`dx.profile` computes FLOPs analytically — no GPU needed, no warmup.

In [ ]:
print('FLOPs by sequence length:')
for seq_len in [64, 128, 256, 512, 1024]:
    flops = count_flops(cfg, seq_len=seq_len, batch_size=1)
    print(f'  seq_len={seq_len:5d}  total={flops.total / 1e9:.3f} GFLOPs')

In [ ]:
# Compare FLOPs across attention types at seq_len=512
attn_cfgs = [
    ('MHA',            dx.ModelConfig(dim=256, n_heads=4, head_size=64, num_blocks=4,
                                      vocab_size=2000, attention='mha')),
    ('GQA kv_heads=2', dx.ModelConfig(dim=256, n_heads=4, head_size=64, num_blocks=4,
                                      vocab_size=2000, attention='gqa', kv_heads=2)),
    ('GQA kv_heads=1', dx.ModelConfig(dim=256, n_heads=4, head_size=64, num_blocks=4,
                                      vocab_size=2000, attention='gqa', kv_heads=1)),
    ('MLA',            dx.ModelConfig(dim=256, n_heads=4, head_size=64, num_blocks=4,
                                      vocab_size=2000, attention='mla')),
]

print('\nFLOPs by attention type (seq_len=512):')
for name, c in attn_cfgs:
    flops = count_flops(c, seq_len=512, batch_size=1)
    n     = dx.ARParadigm(c).num_parameters(dx.ARParadigm(c).build_model(nnx.Rngs(0)))
    print(f'  {name:20s}  {n/1e6:6.2f}M params  {flops.total/1e9:.3f} GFLOPs')

## 2. Wall-Clock Latency

`LatencyTracker` measures actual GPU throughput in tokens/s with warmup.

In [ ]:
tracker = LatencyTracker()
x = jax.random.randint(jax.random.PRNGKey(0), (4, 256), 0, 2000)

# Warmup (forces JIT compilation)
for _ in range(5):
    _ = model(x)

# Measure
for _ in range(20):
    with tracker.measure(n_tokens=4 * 256):
        _ = model(x)

print(tracker.result())

## 3. Parameter Count Across Model Sizes

In [ ]:
size_cfgs = [
    ('Tiny 13M',   dx.ModelConfig(dim=128,  n_heads=2, head_size=64, num_blocks=4,  vocab_size=2000)),
    ('Small 50M',  dx.ModelConfig(dim=256,  n_heads=4, head_size=64, num_blocks=8,  vocab_size=2000)),
    ('Medium 125M',dx.ModelConfig(dim=512,  n_heads=8, head_size=64, num_blocks=12, vocab_size=2000)),
    ('Large 350M', dx.ModelConfig(dim=1024, n_heads=16,head_size=64, num_blocks=24, vocab_size=2000)),
]

print(f'{"Name":15s}  {"Params":>10s}  {"GFLOPs @512":>12s}')
for name, c in size_cfgs:
    p = dx.ARParadigm(c)
    m = p.build_model(nnx.Rngs(0))
    n = p.num_parameters(m)
    f = count_flops(c, seq_len=512, batch_size=1).total / 1e9
    print(f'{name:15s}  {n/1e6:>8.1f}M  {f:>12.2f}')

## 4. Full Benchmark Suite

`BenchmarkSuite.default()` runs a sweep over sequence lengths and batch sizes,
measuring throughput, latency, memory, and perplexity.

In [ ]:
bcfg   = BenchmarkConfig(
    seq_lens    = [64, 128, 256],
    batch_sizes = [1, 4, 16],
    n_warmup    = 3,
    n_measure   = 10,
    eval_batches= 10,
)
report = BenchmarkSuite.default(bcfg).run(paradigm, model, save_csv='/tmp/bench.csv')
print(report.summary())

## 5. Benchmark Across Attention Configs

In [ ]:
import pandas as pd

bcfg_quick = BenchmarkConfig(seq_lens=[128, 256], batch_sizes=[1, 4],
                              n_warmup=2, n_measure=5, eval_batches=5)

rows = []
for name, c in attn_cfgs:
    p = dx.ARParadigm(c)
    m = p.build_model(nnx.Rngs(0))
    rep = BenchmarkSuite.default(bcfg_quick).run(p, m)
    rows.append({'attention': name, **rep.to_dict()})

df_cmp = pd.DataFrame(rows)
print(df_cmp[['attention', 'throughput_tok_s', 'latency_ms_mean']].to_string(index=False))

## 6. Visualization

In [ ]:
import os
from dantinox.visualization import Visualizer
from IPython.display import Image, display

os.makedirs('/tmp/plots', exist_ok=True)
df    = pd.read_csv('/tmp/bench.csv')
paths = Visualizer().render(df, out_dir='/tmp/plots')

for name, path in paths.items():
    print(name)
    display(Image(str(path)))